# EV Energy Consumption - Data Quality Workflow

This notebook follows a connected workflow for understanding energy-consumption data. The original dataset remains unchanged, while a separate copy is created for controlled data-quality experiments.


In [16]:
import pandas as pd
import numpy as np

## Prepare the Tools

The next cell imports the libraries used throughout the workflow: `pandas` for working with tabular data and 
umpy` for controlled random selections.


In [17]:
# STEP 1: LOAD THE ORIGINAL DATASET
##? We import the original CSV file.
##? We keep this dataset unchanged so we can compare it with
##? the dirty and cleaned datasets later.

df_original = pd.read_csv("data.csv")

## Step 1: Load the Original Dataset

The `data.csv` file is loaded into `df_original`. This dataset remains the fixed reference for comparison, so no changes are applied to it.


In [18]:
# Display the number of rows and columns.
# This helps us understand the size of the dataset.

print("Original Dataset Shape:")
print("Rows:", df_original.shape[0])
print("Columns:", df_original.shape[1])

Original Dataset Shape:
Rows: 5000
Columns: 19


## Display the Dataset Size

After loading the data, we check the number of rows and columns to understand the dataset size before starting the quality checks.


In [19]:
# Display the first five rows.
# This helps us understand what the data looks like.

print("\nFirst Five Rows:")
display(df_original.head())


First Five Rows:


,Vehicle_ID,Timestamp,Speed_kmh,Acceleration_ms2,Battery_State_%,Battery_Voltage_V,Battery_Temperature_C,Driving_Mode,Road_Type,Traffic_Condition,Slope_%,Weather_Condition,Temperature_C,Humidity_%,Wind_Speed_ms,Tire_Pressure_psi,Vehicle_Weight_kg,Distance_Travelled_km,Energy_Consumption_kWh
0,1102,2024-01-01 00:00:00,111.507366,-2.773816,30.415148,378.091525,25.314786,2,1,1,6.879446,4,0.741770,42.172533,7.829253,31.112020,1822.967368,20.757508,12.054317
1,1435,2024-01-01 00:01:00,48.612323,-0.796982,97.385534,392.718377,18.240755,1,2,1,-3.007212,4,-3.495516,57.018427,4.495572,31.504366,2091.831914,0.642918,4.488701
2,1860,2024-01-01 00:02:00,108.733320,0.253800,84.912600,398.993495,44.449145,1,1,3,0.029585,1,9.248275,69.028911,5.144489,33.838015,1816.702497,40.842824,11.701377
3,1270,2024-01-01 00:03:00,38.579484,-2.111395,28.777904,358.128273,28.980155,1,2,2,8.271943,3,2.868409,86.638349,4.518283,33.256014,1283.102642,5.305229,7.389266
4,1106,2024-01-01 00:04:00,57.172438,1.477883,29.740160,310.888162,33.184551,2,1,1,2.776814,2,16.750244,27.189185,4.263406,33.579678,2160.350788,5.825926,6.761205


## Preview the Dataset

We display the first five records to understand the structure of the columns and their values before classification or processing.


In [20]:
# Define columns based on their real meaning
categorical_columns = [
    "Driving_Mode",
    "Road_Type",
    "Traffic_Condition",
    "Weather_Condition"
]

identifier_columns = [
    "Vehicle_ID"
]

datetime_columns = [
    "Timestamp"
]

target_column = "Energy_Consumption_kWh"

# Create the classification table
classification = []

for col in df_original.columns:

    if col == target_column:
        feature_type = "Target - Numerical"

    elif col in categorical_columns:
        feature_type = "Categorical"

    elif col in identifier_columns:
        feature_type = "Identifier"

    elif col in datetime_columns:
        feature_type = "Datetime"

    elif pd.api.types.is_numeric_dtype(df_original[col]):
        feature_type = "Numerical"

    else:
        feature_type = "Needs Review"

    classification.append({
        "Column": col,
        "Data Type": str(df_original[col].dtype),
        "Feature Type": feature_type,
        "Unique Values": df_original[col].nunique()
    })

# Display the result
classification_df = pd.DataFrame(classification)

print(classification_df.to_string(index=False))

                Column Data Type       Feature Type  Unique Values
            Vehicle_ID     int64         Identifier            989
             Timestamp    object           Datetime           5000
             Speed_kmh   float64          Numerical           5000
      Acceleration_ms2   float64          Numerical           5000
       Battery_State_%   float64          Numerical           5000
     Battery_Voltage_V   float64          Numerical           5000
 Battery_Temperature_C   float64          Numerical           5000
          Driving_Mode     int64        Categorical              3
             Road_Type     int64        Categorical              3
     Traffic_Condition     int64        Categorical              3
               Slope_%   float64          Numerical           5000
     Weather_Condition     int64        Categorical              4
         Temperature_C   float64          Numerical           5000
            Humidity_%   float64          Numerical           

## Classify the Columns

Each column is classified according to its meaning: identifier, datetime, categorical, numerical, or target variable. A summary table is then created with the data type and number of unique values for every column.


In [21]:
# STEP 2: CHECK THE ORIGINAL DATA QUALITY

# Check missing values in every column.
# Missing values are empty cells that may affect analysis
# and machine learning.

print("\nMissing Values in Original Dataset:")
display(df_original.isnull().sum())


Missing Values in Original Dataset:


Vehicle_ID                0
Timestamp                 0
Speed_kmh                 0
Acceleration_ms2          0
Battery_State_%           0
Battery_Voltage_V         0
Battery_Temperature_C     0
Driving_Mode              0
Road_Type                 0
Traffic_Condition         0
Slope_%                   0
Weather_Condition         0
Temperature_C             0
Humidity_%                0
Wind_Speed_ms             0
Tire_Pressure_psi         0
Vehicle_Weight_kg         0
Distance_Travelled_km     0
Energy_Consumption_kWh    0
dtype: int64

## Step 2: Check Missing Values

We count the missing values in every column of the original dataset. This shows whether the data requires processing before analysis or machine-learning tasks.


In [22]:
# Count exact duplicate rows.
# A duplicate row repeats the same values in every column.

print("\nDuplicate Rows in Original Dataset:")
print(df_original.duplicated().sum())


Duplicate Rows in Original Dataset:
0


## Check Duplicate Rows

We count rows that are completely identical. Duplicate records can repeat the same information and affect the analysis.


In [23]:
# Display summary statistics for numerical columns.
# These statistics help us understand ranges, averages,
# and possible unusual values.

print("\nOriginal Numerical Statistics:")
display(df_original.describe().T.round(2))

# At this point, we have inspected the original dataset
# without changing any of its values.


Original Numerical Statistics:


,count,mean,std,min,25%,50%,75%,max
Vehicle_ID,5000.0,1505.70,290.84,1000.00,1253.00,1510.00,1761.00,1999.00
Speed_kmh,5000.0,59.31,34.52,0.00,29.52,58.65,89.03,119.95
Acceleration_ms2,5000.0,-0.02,1.72,-3.00,-1.49,-0.04,1.47,3.00
Battery_State_%,5000.0,60.21,23.13,20.02,40.09,60.49,80.41,99.99
Battery_Voltage_V,5000.0,351.12,28.97,300.00,326.61,351.51,375.97,399.99
Battery_Temperature_C,5000.0,27.51,10.02,10.00,18.88,27.58,35.92,44.99
Driving_Mode,5000.0,2.01,0.81,1.00,1.00,2.00,3.00,3.00
Road_Type,5000.0,2.00,0.81,1.00,1.00,2.00,3.00,3.00
Traffic_Condition,5000.0,2.01,0.81,1.00,1.00,2.00,3.00,3.00
Slope_%,5000.0,2.50,4.36,-4.99,-1.24,2.45,6.37,10.00


## Check Descriptive Statistics

We display numerical statistics such as count, mean, standard deviation, minimum, and maximum. These statistics help us understand value ranges and identify unusual values without changing the original data.


In [24]:
# STEP 3: CREATE A DIRTY COPY

# We create a separate copy of the original dataset.
# All intentional problems will be added to this copy.
# The original dataset remains unchanged.

df_dirty = df_original.copy()

## Step 3: Create a Dirty Copy

After inspecting the original dataset, we create `df_dirty` as an independent copy. All following data-quality problems are added only to this copy, so `df_original` remains unchanged.


In [25]:
# Create a new DataFrame with additional rows
new_rows = pd.DataFrame([
    {
        "Vehicle_ID": 9991,
        "Timestamp": "2026-01-01 10:00:00",
        "Speed_kmh": None,
        "Acceleration_ms2": 1.2,
        "Battery_State_%": 120,
        "Battery_Voltage_V": 400,
        "Battery_Temperature_C": 30,
        "Driving_Mode": 1,
        "Road_Type": 1,
        "Traffic_Condition": 2,
        "Slope_%": 3,
        "Weather_Condition": 1,
        "Temperature_C": 25,
        "Humidity_%": 50,
        "Wind_Speed_ms": 3,
        "Tire_Pressure_psi": 35,
        "Vehicle_Weight_kg": 1800,
        "Distance_Travelled_km": 10,
        "Energy_Consumption_kWh": 5
    },

    {
        "Vehicle_ID": 9992,
        "Timestamp": "2026-01-01 11:00:00",
        "Speed_kmh": -20,
        "Acceleration_ms2": 2,
        "Battery_State_%": 80,
        "Battery_Voltage_V": 400,
        "Battery_Temperature_C": 32,
        "Driving_Mode": 2,
        "Road_Type": 1,
        "Traffic_Condition": 2,
        "Slope_%": 4,
        "Weather_Condition": 1,
        "Temperature_C": 24,
        "Humidity_%": 45,
        "Wind_Speed_ms": 2,
        "Tire_Pressure_psi": 35,
        "Vehicle_Weight_kg": 1800,
        "Distance_Travelled_km": -5,
        "Energy_Consumption_kWh": 6
    }])

## Add Experimental Rows

Two additional rows are prepared with examples of different data-quality problems, such as missing, negative, or out-of-range values. These rows are intended only for experiments and are not added to the original dataset.


In [26]:
# We use a fixed random seed so the same rows are selected
# whenever we run this code again.

np.random.seed(42)

## Make the Random Process Reproducible

A fixed random seed ensures that the same selections produce the same results whenever the notebook is run again.


In [27]:
# ------------------------------------------------------------
# 3A: INTRODUCE MISSING VALUES
# ------------------------------------------------------------

# We deliberately introduce missing values in selected columns.
# This allows us to demonstrate how missing values are detected
# and handled during the cleaning process.

columns_with_missing_values = [
    "Speed_kmh",
    "Battery_State_%",
    "Temperature_C",
    "Tire_Pressure_psi",
    "Energy_Consumption_kWh"
]

for column in columns_with_missing_values:

    # Check that the column exists before changing it.
    if column in df_dirty.columns:

        # Select approximately 20% of the rows.
        number_of_missing_values = max(
            1, int(len(df_dirty) * 0.20)
        )

        selected_rows = np.random.choice(
            df_dirty.index,
            size=number_of_missing_values,
            replace=False
        )

        # Replace the selected values with NaN.
        df_dirty.loc[selected_rows, column] = np.nan

## 3A: Introduce Missing Values

Selected columns receive missing values in approximately 20% of the rows. This simulates incomplete data so it can later be detected and handled.


In [28]:
# 3B: INTRODUCE DUPLICATE ROWS

# We add 20 existing rows to the end of the dataset.
# These rows become exact duplicates.
# This gives us duplicate records to identify and remove.

number_of_duplicates = min(25, len(df_dirty))

duplicate_rows = df_dirty.sample(
    n=number_of_duplicates,
    random_state=42
)

df_dirty = pd.concat(
    [df_dirty, duplicate_rows],
    ignore_index=True
)

## 3B: Introduce Duplicate Rows

A limited number of existing rows are selected and appended to the dataset. These records become intentional duplicates for testing duplicate detection and removal.


In [29]:
# 3C: INTRODUCE EXTREME VALUES
# We deliberately insert unusual values into selected columns.
# These values are examples for practicing outlier investigation.
# They are not values discovered in the original dataset.

artificial_outliers = {
    "Speed_kmh": 500,
    "Battery_State_%": 150,
    "Battery_Voltage_V": 1000,
    "Temperature_C": 200,
    "Tire_Pressure_psi": 100,
    "Energy_Consumption_kWh": 100
}

for column, unusual_value in artificial_outliers.items():

    if column in df_dirty.columns and len(df_dirty) > 0:

        # Select one row and replace its value with an extreme value.
        selected_row = np.random.choice(df_dirty.index)

        df_dirty.loc[selected_row, column] = unusual_value

## 3C: Introduce Extreme Values

Very large values are inserted into selected columns. These artificial values are used to practice investigating outliers and were not discovered in the original dataset.


In [30]:
# Save the dirty dataset separately.
# This lets us keep the original file unchanged.

df_dirty.to_csv(
    "EV_Energy_Consumption_Dirty.csv",
    index=False
)

print("\nDirty Dataset Created Successfully!")
print("Rows:", df_dirty.shape[0])
print("Columns:", df_dirty.shape[1])


Dirty Dataset Created Successfully!
Rows: 5025
Columns: 19


## Save the Dirty Dataset

The `df_dirty` DataFrame is saved as `EV_Energy_Consumption_Dirty.csv`, and its size is displayed to confirm successful creation. The original dataset remains unchanged.
